# 01 · Experimentos completos
**Proyecto:** Control robótico a partir de datos registrados: Decision Transformer frente a métodos clásicos de RL offline
**Autor:** Daniel Sebastian Velasco Munar · Aprendizaje Profundo 202630 · PUJ

Se entrenan y evalúan los cuatro modelos en los dos robots (Hopper y HalfCheetah) con los tres niveles de datos
(simple, medium y expert) y tres semillas: 18 corridas por modelo.

- MLP, IQL y Decision Transformer se entrenan con **100 mil pasos** cada uno.
- Cada agente se evalúa con 10 episodios en el simulador.
- Cada resultado se guarda en Google Drive apenas termina. Si Colab se desconecta, basta con volver a ejecutar
  todo: las corridas ya hechas se saltan y se sigue donde iba.

**Para que termine más rápido** se puede abrir este mismo notebook en tres pestañas de Colab, cada una con una
semilla distinta en `SEMILLAS_A_CORRER` (sección 2): `[0]`, `[1]` y `[2]`. Cada pestaña guarda su propio archivo y
la sección 7 los junta. Con una sola pestaña y `[0, 1, 2]` también funciona, pero tarda unas tres veces más.

> **Antes de ejecutar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.
> Deja la pestaña abierta y el computador encendido mientras corre.

## 1. Instalación

In [ ]:
# Se instala por partes: si un paquete falla, el error queda a la vista y no frena a los demás.
!python --version
!pip install -q "gymnasium[mujoco]" mujoco
!pip install -q "minari[all]"
!pip install -q d3rlpy transformers imageio
!pip list 2>/dev/null | grep -iE "^(minari|gymnasium|mujoco|d3rlpy|transformers) "

In [ ]:
import os, sys, time, json, re, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.environ["MUJOCO_GL"] = "egl"      # render sin pantalla en Colab

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn
import gymnasium as gym, minari, mujoco, d3rlpy, transformers
from sklearn.ensemble import RandomForestRegressor
from transformers import DecisionTransformerConfig, DecisionTransformerModel
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
for lib in (minari, gym, mujoco, d3rlpy, transformers, torch):
    print(f"{lib.__name__:>13}: {lib.__version__}")

## 2. Configuración y carpeta de resultados
Los resultados quedan en `Mi unidad/proyecto_decision_transformer` de Google Drive.

In [ ]:
ENVS      = ["hopper", "halfcheetah"]
QUALITIES = ["simple", "medium", "expert"]
SEEDS     = [0, 1, 2]
SEMILLAS_A_CORRER = [0, 1, 2]   # en tres pestañas a la vez: [0] en una, [1] en otra y [2] en la otra
MODELS    = ["BC · Random Forest", "BC · MLP", "IQL", "Decision Transformer"]

STEPS         = 100_000   # pasos de entrenamiento de MLP, IQL y Decision Transformer
EVAL_EPISODES = 10        # episodios de evaluación en el simulador por agente
LOG_EVERY     = 1_000     # cada cuántos pasos se guarda la pérdida

# Decision Transformer (mismos valores que en la prueba de viabilidad, tomados del artículo original)
K, RTG_SCALE, BATCH, MAX_EP_LEN = 20, 1000.0, 64, 1000

# Random Forest: usa una muestra del dataset porque con el millón de pasos completo tarda demasiado
RF_SAMPLES, RF_TREES = 100_000, 100

dataset_id = lambda env_name, q: f"mujoco/{env_name}/{q}-v0"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/proyecto_decision_transformer"
else:
    OUT = "resultados"
os.makedirs(f"{OUT}/curvas", exist_ok=True); os.makedirs(f"{OUT}/modelos", exist_ok=True)
REFS_JSON, DATA_CSV = f"{OUT}/referencias.json", f"{OUT}/datos.csv"
res_csv = lambda seed: f"{OUT}/resultados_semilla_{seed}.csv"   # un archivo por semilla para que las pestañas no choquen
print("Resultados en:", OUT)

## 3. Funciones comunes
Todos los agentes tienen la misma interfaz (`reset`, `act`, `update`) y se evalúan con las mismas semillas del
simulador, así la comparación es justa.

In [ ]:
def load_episodes(ds_id):
    ds = minari.load_dataset(ds_id, download=True)
    eps = []
    for ep in ds.iterate_episodes():
        # observations trae un estado de más (el final), por eso se descarta el último
        e = dict(obs=ep.observations[:-1].astype(np.float32), act=ep.actions.astype(np.float32),
                 rew=ep.rewards.astype(np.float32), term=ep.terminations, trunc=ep.truncations)
        e["rtg"] = np.cumsum(e["rew"][::-1])[::-1].astype(np.float32)   # retorno restante
        eps.append(e)
    return ds, eps

def prepare(eps, seed):
    # Partición 90/10 por episodios y estadísticas para normalizar los estados
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(eps)); n_val = max(1, int(0.1 * len(eps)))
    train, val = [eps[i] for i in perm[n_val:]], [eps[i] for i in perm[:n_val]]
    cat = lambda es, k: np.concatenate([e[k] for e in es])
    X, Y = cat(train, "obs"), cat(train, "act")
    mean, std = X.mean(0), X.std(0) + 1e-6
    for e in train + val:
        # se normaliza una sola vez aquí y no en cada lote del Decision Transformer
        e["obs_n"] = ((e["obs"] - mean) / std).astype(np.float32); e["rtg_s"] = e["rtg"] / RTG_SCALE
    return dict(train=train, val=val, X=X, Y=Y, Xv=cat(val, "obs"), Yv=cat(val, "act"), mean=mean, std=std,
                norm=lambda s: ((s - mean) / std).astype(np.float32), sd=X.shape[1], ad=Y.shape[1])

class SimpleAgent:
    # Agente sin memoria: la acción depende solo del estado actual
    def __init__(self, fn): self.fn = fn
    def reset(self): pass
    def update(self, r): pass
    def act(self, obs): return np.clip(self.fn(obs), -1, 1)

def evaluate(env, agent, n=EVAL_EPISODES, seed0=10_000):
    rets, lens = [], []
    for k in range(n):
        obs, _ = env.reset(seed=seed0 + k); agent.reset(); done, R, L = False, 0.0, 0
        while not done:
            obs, r, term, trunc, _ = env.step(agent.act(obs)); agent.update(r)
            R += r; L += 1; done = term or trunc
        rets.append(R); lens.append(L)
    return np.array(rets), np.array(lens)

normalized = lambda env_name, R: 100 * (R - REFS[env_name]["random"]) / (REFS[env_name]["expert"] - REFS[env_name]["random"])

## 4. Datos y referencias del puntaje normalizado
Se descargan los seis datasets y se resume cada uno. El puntaje normalizado es
$100 \cdot \frac{R - R_{aleatorio}}{R_{experto} - R_{aleatorio}}$, donde $R_{experto}$ es el retorno medio del dataset
expert y $R_{aleatorio}$ el de un agente que actúa al azar.

In [ ]:
REFS = json.load(open(REFS_JSON)) if os.path.exists(REFS_JSON) else {}
data_rows = []
for env_name in ENVS:
    for q in QUALITIES:
        ds, eps = load_episodes(dataset_id(env_name, q))
        R = np.array([e["rew"].sum() for e in eps]); T = np.array([len(e["act"]) for e in eps])
        data_rows.append({"Robot": env_name, "Datos": q, "Episodios": len(eps), "Pasos": int(T.sum()),
                          "Retorno medio": R.mean(), "Retorno mín": R.min(), "Retorno máx": R.max(),
                          "Termina por caída": np.mean([e["term"][-1] for e in eps])})
        if q == "expert" and env_name not in REFS:
            env = ds.recover_environment(); env.action_space.seed(0)
            rand_R, _ = evaluate(env, SimpleAgent(lambda o: env.action_space.sample()))
            REFS[env_name] = {"random": float(rand_R.mean()), "expert": float(R.mean())}
            env.close()
        del eps
json.dump(REFS, open(REFS_JSON, "w"), indent=2)
data_info = pd.DataFrame(data_rows); data_info.to_csv(DATA_CSV, index=False)
display(data_info.round(2))
print("Referencias:", json.dumps(REFS, indent=1))

## 5. Los cuatro modelos

### Clonación de comportamiento con Random Forest (modelo tradicional)

In [ ]:
def train_rf(D, seed, env_name):
    idx = np.random.default_rng(seed).choice(len(D["X"]), size=min(RF_SAMPLES, len(D["X"])), replace=False)
    rf = RandomForestRegressor(n_estimators=RF_TREES, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=seed)
    rf.fit(D["X"][idx], D["Y"][idx])
    rf.set_params(n_jobs=1)   # para un solo estado es más rápido sin paralelismo
    val_mse = np.mean((rf.predict(D["Xv"]) - D["Yv"])**2)
    return SimpleAgent(lambda o: rf.predict(o[None])[0]), val_mse, []

### Clonación de comportamiento con MLP

In [ ]:
def train_mlp(D, seed, env_name):
    torch.manual_seed(seed)
    mlp = nn.Sequential(nn.Linear(D["sd"], 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                        nn.Linear(256, D["ad"]), nn.Tanh()).to(DEVICE)
    opt = torch.optim.Adam(mlp.parameters(), lr=3e-4)
    Xt = torch.tensor(D["norm"](D["X"]), device=DEVICE); Yt = torch.tensor(D["Y"], device=DEVICE)
    curve, acc = [], 0.0
    for step in range(STEPS):
        b = torch.randint(0, len(Xt), (256,), device=DEVICE)
        loss = ((mlp(Xt[b]) - Yt[b])**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        acc += loss.detach()
        if (step + 1) % LOG_EVERY == 0:
            curve.append((step + 1, "pérdida", (acc / LOG_EVERY).item())); acc = 0.0
    mlp.eval()
    with torch.no_grad():
        val_mse = ((mlp(torch.tensor(D["norm"](D["Xv"]), device=DEVICE)) - torch.tensor(D["Yv"], device=DEVICE))**2).mean().item()

    @torch.no_grad()
    def policy(o): return mlp(torch.tensor(D["norm"](o)[None], device=DEVICE))[0].cpu().numpy()
    return SimpleAgent(policy), val_mse, curve

### Implicit Q-Learning (RL offline con función Q, d3rlpy)

In [ ]:
def train_iql(D, seed, env_name):
    d3rlpy.seed(seed)
    cat = lambda k: np.concatenate([e[k] for e in D["train"]])
    data = d3rlpy.dataset.MDPDataset(
        observations=cat("obs"), actions=cat("act"), rewards=cat("rew"),
        terminals=cat("term").astype(np.float32), timeouts=cat("trunc").astype(np.float32))
    iql = d3rlpy.algos.IQLConfig(
        expectile=0.7, weight_temp=3.0, max_weight=100.0,
        observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
        reward_scaler=d3rlpy.preprocessing.ReturnBasedRewardScaler(multiplier=1000.0),
    ).create(device="cuda:0" if DEVICE == "cuda" else "cpu:0")
    per_epoch = LOG_EVERY * 10
    log = iql.fit(data, n_steps=STEPS, n_steps_per_epoch=per_epoch, show_progress=False,
                  logger_adapter=d3rlpy.logging.NoopAdapterFactory())
    curve = [(ep * per_epoch, name, m[key]) for ep, m in log
             for name, key in (("pérdida crítico", "critic_loss"), ("pérdida actor", "actor_loss"))]
    val_mse = np.mean((iql.predict(D["Xv"].astype(np.float32)) - D["Yv"])**2)
    return SimpleAgent(lambda o: iql.predict(o[None].astype(np.float32))[0]), val_mse, curve

### Decision Transformer (Hugging Face)
Al evaluar se le pide como retorno objetivo el retorno medio del dataset expert de cada robot, el mismo para los
tres niveles de datos.

In [ ]:
def get_batch(D, eps, rng, probs=None):
    # Segmentos de longitud K con relleno a la izquierda y máscara de atención
    s = np.zeros((BATCH, K, D["sd"]), np.float32); a = np.zeros((BATCH, K, D["ad"]), np.float32)
    r = np.zeros((BATCH, K, 1), np.float32); g = np.zeros((BATCH, K, 1), np.float32)
    ts = np.zeros((BATCH, K), np.int64); m = np.zeros((BATCH, K), np.float32)
    for b, i in enumerate(rng.choice(len(eps), size=BATCH, p=probs)):
        e = eps[i]; T = len(e["act"]); si = rng.integers(0, T); L = min(K, T - si); sl = slice(si, si + L)
        s[b, K-L:] = e["obs_n"][sl]; a[b, K-L:] = e["act"][sl]; r[b, K-L:, 0] = e["rew"][sl]
        g[b, K-L:, 0] = e["rtg_s"][sl]; ts[b, K-L:] = np.arange(si, si + L); m[b, K-L:] = 1
    to = lambda x: torch.from_numpy(x).to(DEVICE, non_blocking=True)
    return to(s), to(a), to(r), to(g), to(ts), to(m)

class DTAgent:
    # Genera acciones de forma autorregresiva con los últimos K pasos de historia
    def __init__(self, model, D, target_return): self.model, self.D, self.target = model, D, target_return
    def reset(self):
        self.S, self.A, self.R = [], [], []; self.G = [self.target / RTG_SCALE]; self.t = 0
    @torch.no_grad()
    def act(self, obs):
        sd, ad = self.D["sd"], self.D["ad"]
        self.S.append(self.D["norm"](obs)); self.A.append(np.zeros(ad, np.float32)); self.R.append(0.0)
        L = min(K, len(self.S)); pad = K - L
        def seq(x, dim):
            x = np.asarray(x[-L:], np.float32).reshape(L, dim)
            return np.concatenate([np.zeros((pad, dim), np.float32), x])
        ts = np.concatenate([np.zeros(pad, np.int64), np.minimum(np.arange(self.t - L + 1, self.t + 1), MAX_EP_LEN - 1)])
        m = np.concatenate([np.zeros(pad), np.ones(L)]).astype(np.float32)
        to = lambda x: torch.tensor(x[None], device=DEVICE)
        out = self.model(states=to(seq(self.S, sd)), actions=to(seq(self.A, ad)), rewards=to(seq(self.R, 1)),
                         returns_to_go=to(seq(self.G, 1)), timesteps=to(ts), attention_mask=to(m), return_dict=True)
        action = out.action_preds[0, -1].cpu().numpy(); self.A[-1] = action
        return action
    def update(self, r):
        self.R[-1] = r; self.G.append(self.G[-1] - r / RTG_SCALE); self.t += 1

def train_dt(D, seed, env_name):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    probs = np.array([len(e["act"]) for e in D["train"]], dtype=float); probs /= probs.sum()
    config = DecisionTransformerConfig(state_dim=D["sd"], act_dim=D["ad"], hidden_size=128, n_layer=3, n_head=1,
                                       n_inner=512, activation_function="relu", max_ep_len=MAX_EP_LEN,
                                       resid_pdrop=0.1, attn_pdrop=0.1, action_tanh=True)
    dt = DecisionTransformerModel(config).to(DEVICE)

    def loss_fn(batch):
        s, a, r, g, ts, m = batch
        out = dt(states=s, actions=a, rewards=r, returns_to_go=g, timesteps=ts, attention_mask=m, return_dict=True)
        return (((out.action_preds - a)**2).mean(-1) * m).sum() / m.sum()

    opt = torch.optim.AdamW(dt.parameters(), lr=1e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda step: min(1.0, (step + 1) / 10_000))   # calentamiento
    curve, acc = [], 0.0
    dt.train()
    for step in range(STEPS):
        loss = loss_fn(get_batch(D, D["train"], rng, probs))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(dt.parameters(), 0.25); opt.step(); sched.step()
        acc += loss.detach()
        if (step + 1) % LOG_EVERY == 0:
            curve.append((step + 1, "pérdida", (acc / LOG_EVERY).item())); acc = 0.0
    dt.eval()
    with torch.no_grad():
        val_mse = np.mean([loss_fn(get_batch(D, D["val"], rng)).item() for _ in range(50)])
    return DTAgent(dt, D, REFS[env_name]["expert"]), val_mse, curve

TRAINERS = {"BC · Random Forest": train_rf, "BC · MLP": train_mlp, "IQL": train_iql, "Decision Transformer": train_dt}

## 6. Corridas
Se recorre primero la semilla, así después de la primera vuelta ya hay un resultado para cada robot y nivel de datos.
Cada fila se guarda en `resultados_semilla_N.csv`, cada curva de pérdida en la carpeta `curvas` y cada
Decision Transformer entrenado en la carpeta `modelos`.

In [ ]:
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
done = set()
for seed in SEMILLAS_A_CORRER:
    if os.path.exists(res_csv(seed)):
        prev = pd.read_csv(res_csv(seed))
        done |= set(zip(prev["Robot"], prev["Datos"], prev["Semilla"], prev["Modelo"]))
total = len(SEMILLAS_A_CORRER) * len(ENVS) * len(QUALITIES) * len(MODELS)
print(f"Ya hechas: {len(done)} de {total}")

t_start = time.time(); n_new = 0
for seed in SEMILLAS_A_CORRER:
    for env_name in ENVS:
        for q in QUALITIES:
            pending = [m for m in MODELS if (env_name, q, seed, m) not in done]
            if not pending:
                continue
            ds, eps = load_episodes(dataset_id(env_name, q))
            env = ds.recover_environment(); D = prepare(eps, seed)
            for model in pending:
                t0 = time.time(); agent, val_mse, curve = TRAINERS[model](D, seed, env_name); t_train = time.time() - t0
                t0 = time.time(); rets, lens = evaluate(env, agent); t_eval = time.time() - t0
                if model == "Decision Transformer":
                    # se guarda para probar después otros retornos objetivo sin volver a entrenar
                    torch.save({"pesos": agent.model.state_dict(), "config": agent.model.config.to_dict(),
                                "media": D["mean"], "desv": D["std"]}, f"{OUT}/modelos/dt_{env_name}_{q}_{seed}.pt")
                pd.DataFrame(curve, columns=["Paso", "Métrica", "Valor"]).to_csv(
                    f"{OUT}/curvas/{env_name}_{q}_{seed}_{slug(model)}.csv", index=False)
                row = {"Robot": env_name, "Datos": q, "Semilla": seed, "Modelo": model,
                       "Retorno medio": rets.mean(), "Desv.": rets.std(),
                       "Puntaje normalizado": normalized(env_name, rets.mean()), "Duración media": lens.mean(),
                       "MSE validación": val_mse, "Entrenamiento (s)": t_train, "Evaluación (s)": t_eval,
                       "Retornos": json.dumps(np.round(rets, 1).tolist())}
                pd.DataFrame([row]).to_csv(res_csv(seed), mode="a", header=not os.path.exists(res_csv(seed)), index=False)
                done.add((env_name, q, seed, model)); n_new += 1
                eta = (time.time() - t_start) / n_new * (total - len(done)) / 3600
                print(f"[{len(done):>2}/{total}] {env_name:<11} {q:<6} semilla {seed} · {model:<20} "
                      f"puntaje {row['Puntaje normalizado']:6.1f} · {(t_train + t_eval)/60:5.1f} min · faltan ≈ {eta:.1f} h")
            env.close(); del D, eps; torch.cuda.empty_cache()
print("Listo: semillas", SEMILLAS_A_CORRER, "terminadas")

## 7. Resultados
Esta sección solo lee los archivos guardados, así que se puede ejecutar en cualquier momento, incluso con
corridas a medias.

In [ ]:
files = [res_csv(s) for s in SEEDS if os.path.exists(res_csv(s))]
res = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"Corridas terminadas: {len(res)} de {len(SEEDS) * len(ENVS) * len(QUALITIES) * len(MODELS)}")
res["Datos"] = pd.Categorical(res["Datos"], QUALITIES, ordered=True)
res["Modelo"] = pd.Categorical(res["Modelo"], MODELS, ordered=True)
summary = res.groupby(["Robot", "Datos", "Modelo"], observed=True)["Puntaje normalizado"].agg(["mean", "std", "count"])
table = (summary["mean"].round(1).astype(str) + " ± " + summary["std"].fillna(0).round(1).astype(str)).unstack("Modelo")
print("Puntaje normalizado (media ± desviación entre semillas)")
display(table)

fig, axes = plt.subplots(1, len(ENVS), figsize=(13, 4), sharey=True)
w = 0.2; x = np.arange(len(QUALITIES))
for ax, env_name in zip(np.atleast_1d(axes), ENVS):
    for j, model in enumerate(MODELS):
        s = summary.loc[env_name].xs(model, level="Modelo").reindex(QUALITIES) if env_name in summary.index else None
        if s is None: continue
        ax.bar(x + (j - 1.5) * w, s["mean"], w, yerr=s["std"].fillna(0), capsize=3, label=model)
    ax.set_xticks(x); ax.set_xticklabels(QUALITIES); ax.set_title(env_name.capitalize()); ax.set_xlabel("Nivel de datos")
np.atleast_1d(axes)[0].set_ylabel("Puntaje normalizado"); np.atleast_1d(axes)[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/puntajes.png", dpi=150); plt.show()

tiempos = res.groupby("Modelo", observed=True)[["Entrenamiento (s)", "Evaluación (s)", "MSE validación"]].mean()
print("Promedios por modelo"); display(tiempos.round(3))

In [ ]:
# Curvas de pérdida del Decision Transformer y de la MLP (promedio entre semillas)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, model in zip(axes, ["Decision Transformer", "BC · MLP"]):
    for env_name in ENVS:
        for q in QUALITIES:
            files = [f"{OUT}/curvas/{env_name}_{q}_{s}_{slug(model)}.csv" for s in SEEDS]
            curves = [pd.read_csv(f) for f in files if os.path.exists(f)]
            if not curves: continue
            c = pd.concat(curves).groupby("Paso")["Valor"].mean()
            ax.plot(c.index, c.values, label=f"{env_name} · {q}")
    ax.set_title(model); ax.set_xlabel("Paso"); ax.set_ylabel("MSE"); ax.set_yscale("log")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT}/curvas.png", dpi=150); plt.show()